In [1]:
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.1/320.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.9 MB/s eta 0:00:00


In [2]:
!pip install segmentation-models-pytorch

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 25.7 MB/s eta 0:00:00
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16424 sha256=4f430aee9db973944462eb126fb6a1edceb7b2298ca728512390800cded76d68
  Stored in directory: /root/.cache/pip/wheels/03/3f/e9/911b1bc46869644912bda90a56bcf7b960f20b5187feea3baf
  Created wheel for pretrainedmodels: filename=pretrainedmodels-0.7.4-py3-none-any.whl size=60944 sha256=a42bc2f21b356e11fcdd26ad1a1842d61481711942abbf9d408007ad285ee40c
  Stored in directory: /root/.cache/pip/wheels/35/cb/a5/8f534c60142835bfc889f9a482e4a67e0b817032d9c6883b64
Successfully built efficientnet-pytorch

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pwd

/content


In [5]:
%cd drive/MyDrive/CS331/Dataset-FoodSeg103

/content/drive/.shortcut-targets-by-id/1IlgUExhirsyB2WnMMnEhPIkwSvFC8vM6/CS331/Dataset-FoodSeg103


In [6]:
import gradio as gr
import os
import json
import base64
import io
import zlib
from collections import defaultdict
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2
from segmentation_models_pytorch import Unet
from PIL import Image

from torchvision.models.segmentation import fcn_resnet50

In [7]:
def get_filename(file_path):
  return os.path.basename(file_path)

In [8]:
def calculate_pixel_accuracy(prediction, ground_truth):
    """
    Calculate pixel accuracy between prediction and ground truth.

    Args:
    prediction (torch.Tensor or np.ndarray): Predicted segmentation mask
    ground_truth (torch.Tensor or np.ndarray): Ground truth segmentation mask

    Returns:
    float: Pixel accuracy
    """
    # Convert to numpy if tensors
    if torch.is_tensor(prediction):
        prediction = prediction.cpu().numpy()
    if torch.is_tensor(ground_truth):
        ground_truth = ground_truth.cpu().numpy()

    # Ensure the arrays are of the same shape
    assert prediction.shape == ground_truth.shape, "Prediction and ground truth must have the same shape"

    # Calculate pixel-wise accuracy
    correct_pixels = np.sum(prediction == ground_truth)
    total_pixels = prediction.size

    return correct_pixels / total_pixels

In [9]:
def calculate_mean_iou(prediction, ground_truth, num_classes=104, ignore_background=False):
    """
    Calculate mean Intersection over Union (IoU) for all classes,
    only considering classes with non-zero IoU.

    Args:
    prediction (torch.Tensor or np.ndarray): Predicted segmentation mask
    ground_truth (torch.Tensor or np.ndarray): Ground truth segmentation mask
    num_classes (int): Total number of classes
    ignore_background (bool): Whether to ignore the background class in mean IoU calculation

    Returns:
    float: Mean IoU across non-zero classes
    dict: IoU for each class
    """
    # Convert to numpy if tensors
    if torch.is_tensor(prediction):
        prediction = prediction.cpu().numpy()
    if torch.is_tensor(ground_truth):
        ground_truth = ground_truth.cpu().numpy()

    # Ensure the arrays are of the same shape
    assert prediction.shape == ground_truth.shape, "Prediction and ground truth must have the same shape"

    # Calculate IoU for each class
    iou_per_class = {}
    for class_id in range(num_classes):
        # Create binary masks for the current class
        pred_class_mask = (prediction == class_id)
        true_class_mask = (ground_truth == class_id)

        # Calculate intersection and union
        intersection = np.logical_and(pred_class_mask, true_class_mask).sum()
        union = np.logical_or(pred_class_mask, true_class_mask).sum()

        # Calculate IoU, avoiding division by zero
        iou = intersection / (union + 1e-6) if union > 0 else 0
        iou_per_class[class_id] = iou

    # Collect non-zero IoU values
    if ignore_background:
        # Exclude background class (usually class 0)
        iou_values = [iou for cls, iou in iou_per_class.items() if cls != 0 and iou > 0]
    else:
        iou_values = [iou for iou in iou_per_class.values() if iou > 0]

    # Calculate mean IoU only for non-zero classes
    mean_iou = np.mean(iou_values) if iou_values else 0

    return mean_iou, iou_per_class

In [10]:
def calculate_dice_coefficient(prediction, ground_truth, num_classes=104, ignore_background=False):
    """
    Calculate Dice Coefficient (F1 score) for all classes,
    only considering classes with non-zero Dice Coefficient.

    Args:
    prediction (torch.Tensor or np.ndarray): Predicted segmentation mask
    ground_truth (torch.Tensor or np.ndarray): Ground truth segmentation mask
    num_classes (int): Total number of classes
    ignore_background (bool): Whether to ignore the background class in Dice Coefficient calculation

    Returns:
    float: Mean Dice Coefficient across non-zero classes
    dict: Dice Coefficient for each class
    """
    # Convert to numpy if tensors
    if torch.is_tensor(prediction):
        prediction = prediction.cpu().numpy()
    if torch.is_tensor(ground_truth):
        ground_truth = ground_truth.cpu().numpy()

    # Ensure the arrays are of the same shape
    assert prediction.shape == ground_truth.shape, "Prediction and ground truth must have the same shape"

    # Calculate Dice Coefficient for each class
    dice_per_class = {}
    for class_id in range(num_classes):
        # Create binary masks for the current class
        pred_class_mask = (prediction == class_id)
        true_class_mask = (ground_truth == class_id)

        # Calculate intersection, prediction, and ground truth pixels
        intersection = np.logical_and(pred_class_mask, true_class_mask).sum()
        pred_pixels = pred_class_mask.sum()
        true_pixels = true_class_mask.sum()

        # Calculate Dice Coefficient, avoiding division by zero
        dice = (2 * intersection) / (pred_pixels + true_pixels + 1e-6)
        dice_per_class[class_id] = dice

    # Collect non-zero Dice Coefficient values
    if ignore_background:
        # Exclude background class (usually class 0)
        dice_values = [dice for cls, dice in dice_per_class.items() if cls != 0 and dice > 0]
    else:
        dice_values = [dice for dice in dice_per_class.values() if dice > 0]

    # Calculate mean Dice Coefficient only for non-zero classes
    mean_dice = np.mean(dice_values) if dice_values else 0

    return mean_dice, dice_per_class

In [11]:
def get_annpath(root_dir, file_name):
  ann_name= os.path.splitext(file_name)[0]+'.jpg.json'
  ann_path = os.path.join(root_dir, 'ann', ann_name)
  return ann_path

In [12]:
def get_groundtruth(ann_path, dataset, target_shape):
  with open(ann_path,'r') as f:
    annotation = json.load(f)
  ground_truth = np.zeros((annotation['size']['height'], annotation['size']['width']), dtype=np.int32)
  for obj in annotation['objects']:
      original_class_id = obj['classId']
      mapped_class_id = dataset.class_mapping[original_class_id] ###

      bitmap_data = obj['bitmap']['data']
      origin = obj['bitmap']['origin']
      obj_mask = dataset.decode_bitmap(bitmap_data, origin, annotation['size'])
      ground_truth[obj_mask > 0] = mapped_class_id

  ground_truth_pil = Image.fromarray(ground_truth.astype(np.uint8))
  ground_truth_resized = np.array(ground_truth_pil.resize(target_shape[::-1], Image.Resampling.NEAREST))
  return ground_truth_resized

In [13]:
def get_classNames(segment, class_names):
  unique_idx = np.unique(segment)
  return [ class_names.get(str(idx)) for idx in unique_idx]


In [14]:
from torch.utils.data import Dataset
class FoodSegDataset(Dataset):
  def __init__(self, root_dir, target_size=(224,224), split='train', transform=None):
    self.root_dir = root_dir
    self.split = split
    self.transform = transform
    self.img_dir = os.path.join(root_dir, split, 'img')
    self.ann_dir = os.path.join(root_dir, split, 'ann')
    self.images = sorted(os.listdir(self.img_dir))
    self.target_size = target_size

    self.class_mapping = {'background': 0}
    self.reverse_mapping = {0: 'background'}
    self.create_class_mapping()

  def create_class_mapping(self):
    class_ids = set()
    for img_name in self.images:
      ann_name = os.path.splitext(img_name)[0]+'.jpg.json'
      ann_path = os.path.join(self.ann_dir, ann_name)
      with open(ann_path, 'r') as f:
        annotation = json.load(f)

      for obj in annotation['objects']:
        class_ids.add(obj['classId'])

      for idx, classId in enumerate( sorted(class_ids), start=1 ): # excluce background class
        self.class_mapping[classId] = idx
        self.reverse_mapping[idx] = classId
      self.num_classes = len(self.class_mapping) # now 104 with backbround
      #print(f"Total classes including background: {self.num_classes}")

  def decode_bitmap(self,bitmap_data, origin, size):
    try:
      # Remove whitespace and add padding if necessary
      bitmap_data = bitmap_data.strip()
      padding = len( bitmap_data) % 4
      if padding:
        bitmap_data += '=' * (4-padding)

      # Decode base64 data
      decoded_data = base64.b64decode(bitmap_data)

      try:
        decoded_data = zlib.decompress(decoded_data)
      except zlib.error:
        pass

      stream = io.BytesIO(decoded_data)
      bitmap_image = Image.open(stream)
      mask = np.array(bitmap_image)

      if len(mask.shape) == 3:
        mask = mask[:, :, 0]

      full_mask = np.zeros((size['height'], size['width']), dtype=np.int32)

      h, w = mask.shape
      x, y = origin
      x_end = min(x + w, size['width'])
      y_end = min(y + h, size['height'])
      w = x_end - x
      h = y_end - y

      full_mask[y:y+h, x:x+w] = mask[:h, :w]

      return full_mask

    except Exception as e:
      print(f"Error decoding bitmap: {str(e)}")
      return np.zeros((size['height'], size['width']), dtype=np.int32)

  def __getitem__(self, idx):
    try:
      # Load image
      img_name = self.images[idx]
      img_path = os.path.join(self.img_dir, img_name)
      image = Image.open(img_path).convert('RGB')

      # Load annotation
      ann_name = os.path.splitext(img_name)[0]+'.jpg.json'
      ann_path = os.path.join(self.ann_dir, ann_name)

      with open(ann_path, 'r') as f:
        annotation = json.load(f)

      # Create segmentation mask
      mask = np.zeros((annotation['size']['height'], annotation['size']['width']), dtype=np.int32)

      # Fill mask with mapped class labels
      for obj in annotation['objects']:
        original_classId = obj['classId']
        mapped_classId = self.class_mapping[original_classId]
        bitmap_data = obj['bitmap']['data']
        origin = obj['bitmap']['origin']

        obj_mask = self.decode_bitmap(bitmap_data, origin, annotation['size'])
        mask[obj_mask > 0] = mapped_classId

      # Convert mask to PIL Image for resizing
      mask_pil = Image.fromarray(mask.astype(np.uint8))

      # Resize both image and mask to target size
      image = image.resize( self.target_size[::-1], Image.Resampling.BILINEAR ) # reverse order of size
      mask_pil = mask_pil.resize(self.target_size[::-1], Image.Resampling.NEAREST)

      if self.transform:
        image = self.transform(image)

      # Convert mask back to tensor
      mask = torch.from_numpy(np.array(mask_pil)).long()
      return image, mask

    except Exception as e:
      print(f"Error processing item {idx}: {str(e)}")
      dummy_image = torch.zeros((3, *self,target_size))
      dummy_mask = torch.zeros(self.target_size, dtype=torch.long)
      return dummy_image, dummy_mask

  def __len__(self):
    return len(self.images)

In [15]:
def load_model(model_name, weight_path, device, num_classes=104):
  if model_name == 'FCN+Resnet50':
    model = fcn_resnet50(num_classes= num_classes)
  elif model_name == 'UNet':
    model = Unet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=104)
  else: ## DeepLabV3
    model = torch.hub.load('pytorch/vision:v0.10.0', 'deeplabv3_resnet50', pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)

  checkpoint = torch.load(weight_path, map_location=device)
  if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
  else:
    model.load_state_dict(checkpoint)
  model = model.to(device)

  return model

In [16]:
def get_prediction_dict(model, image_tensor, device):
  model.eval()
  with torch.no_grad():
    image_tensor = image_tensor.unsqueeze(0).to(device)
    output = model(image_tensor)['out']

    probabilites = F.softmax(output, dim=1)
    predictions = torch.argmax(probabilites, dim=1)
  return predictions[0].cpu().numpy()

In [17]:
def get_prediction_tensor(model, image_tensor, device):
  model.eval()
  with torch.no_grad():
    image_tensor = image_tensor.unsqueeze(0).to(device)
    output = model(image_tensor)

    probabilites = F.softmax(output, dim=1)
    predictions = torch.argmax(probabilites, dim=1)
  return predictions[0].cpu().numpy()

In [18]:
def create_colored_mask(prediction, dataset, class_names):
  """Create a colored segmentation mask with class names"""
  # Create colormap with consistent colors
  colors = plt.cm.get_cmap('tab20')(np.linspace(0,1, dataset.num_classes))
  colors = (colors[:, :3]*255).astype(np.uint8) # colors shape: [num_classes, 3]

  # Make background color blue
  colors[0] = [0,0,255] #RGB

  colored_mask = colors[prediction]
  return colored_mask, colors

In [19]:
def overlay_mask(image, colored_mask, alpha = 0.5):
  """Overlay the colored mask on the original image"""
  image_np = np.array(image).astype(np.uint8)
  colored_mask = colored_mask.astype(np.uint8)
  if image_np.shape != colored_mask.shape:
    colored_mask = cv2.resize(colored_mask, (image_np.shape[1], image_np.shape[0]))
  overlay = cv2.addWeighted(image_np, 1-alpha, colored_mask, alpha, 0)
  return overlay

In [20]:
def load_image(image_path, target_size=(384, 512)):
    """Load and preprocess an image for inference."""
    image = Image.open(image_path).convert('RGB')
    image = image.resize(target_size[::-1], Image.Resampling.BILINEAR)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    image_tensor = transform(image)
    return image_tensor, image

In [21]:
def predict_segment(image_path, model, device, target_shape ,dataset, class_names):
  if type(model).__name__ == 'Unet':
    get_prediction = get_prediction_tensor
  else:
    get_prediction = get_prediction_dict
  print("oke",type(model).__name__ )

  image_tensor, original_image = load_image(image_path, target_shape)
  prediction = get_prediction(model, image_tensor, device)
  colored_mask, colors = create_colored_mask(prediction, dataset, class_names)
  overlay = overlay_mask(original_image, colored_mask)
  return prediction, colored_mask, overlay

In [22]:

class_mappingFile= 'class_names.json'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

with open(class_mappingFile, 'r') as f:
  class_names= json.load(f)

train_dataset = torch.load('train_dataset.pth')

# Placeholder function for evaluating metrics
def gr_wrapper(image_path, output_type, metric):

    root_dir='test'
    target_shape_1 = (224, 224)
    target_shape_2 = (384, 512)

    file_name= get_filename(image_path)

    ann_path= get_annpath(root_dir, file_name)

    print('#########\nGet filepath done\n#########')

    ground_truth_resized_1 = get_groundtruth(ann_path, train_dataset, target_shape_1)
    ground_truth_resized_2 = get_groundtruth(ann_path, train_dataset, target_shape_2)

    print('Get groundtruth done\n#########')

    true_classes = get_classNames(ground_truth_resized_1, class_names)




    metric_labels = {
        "Pixel Accuracy": "Pixel Accuracy Score",
        "Mean IoU": "Mean IoU Score",
        "Dice Coefficient": "Dice Coefficient Score",
    }

    model1= load_model(model_name='FCN+Resnet50', weight_path='best_fcnresnet50.pth', device=device, num_classes=104)
    model2= load_model(model_name='UNet', weight_path='best_food_segmentation_model_unet.pth', device=device, num_classes=104)
    model3= load_model(model_name='DeepLabV3', weight_path='deeplabV3_30epochs.pth', device=device, num_classes=104)

    print('Load models done\n#########')

    predict1, colored_mask1, overlay1 = predict_segment(image_path, model1, device, target_shape_2, train_dataset, class_names)
    predict2, colored_mask2, overlay2 = predict_segment(image_path, model2, device, target_shape_2, train_dataset, class_names)
    predict3, colored_mask3, overlay3 = predict_segment(image_path, model3, device, target_shape_1, train_dataset, class_names)

    print('Predict segment done\n#########')

    if output_type=='Segmentation Mask':
      pred_model_1= colored_mask1
      pred_model_2= colored_mask2
      pred_model_3= colored_mask3
    else:
      pred_model_1= overlay1
      pred_model_2= overlay2
      pred_model_3= overlay3


    if metric == 'Pixel Accuracy':
      eval_fn = calculate_pixel_accuracy
    elif metric == 'Mean IoU':
      eval_fn= calculate_mean_iou # calculate_mean_iou(prediction, ground_truth, num_classes=104, ignore_background=False)
    else:
      eval_fn= calculate_dice_coefficient # calculate_dice_coefficient(prediction, ground_truth, num_classes=104, ignore_background=False)

    score_model_1 = eval_fn(predict1, ground_truth_resized_2 )
    score_model_2 = eval_fn(predict2, ground_truth_resized_2 )
    score_model_3 = eval_fn(predict3, ground_truth_resized_1)

    if metric!='Pixel Accuracy': # Only take global score
      score_model_1 = score_model_1[0]
      score_model_2 = score_model_2[0]
      score_model_3 = score_model_3[0]
    print('Calculate score done')

    class_model_1 = get_classNames(predict1, class_names)
    class_model_2 = get_classNames(predict2, class_names)
    class_model_3 = get_classNames(predict3, class_names)

    print('Get predict classes done')

    return (true_classes,
            pred_model_1, f"{metric_labels[metric]}: {score_model_1:.2f}", class_model_1,
            pred_model_2, f"{metric_labels[metric]}: {score_model_2:.2f}", class_model_2,
            pred_model_3, f"{metric_labels[metric]}: {score_model_3:.2f}", class_model_3)




<ipython-input-22-bf10beae07ab>:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load('train_dataset.pth')


In [26]:
test = gr_wrapper("/content/drive/MyDrive/CS331/Dataset-FoodSeg103/test/img/00007117.jpg", "Segmentation Mask", "Pixel Accuracy")

#########
Get filepath done
#########
Get groundtruth done
#########


<ipython-input-15-7adaaaf2f7ac>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weight_path, map_location=device)
Using cache found in /root/.cache

Load models done
#########
oke FCN


<ipython-input-18-73b4c97fb18d>:4: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20')(np.linspace(0,1, dataset.num_classes))


oke Unet
oke DeepLabV3
Predict segment done
#########
Calculate score done
Get predict classes done


In [43]:
fnc = {
    "Pixel Accuracy": [],
    "Mean IoU": [],
    "Dice Coefficient": []
}

un = {
    "Pixel Accuracy": [],
    "Mean IoU": [],
    "Dice Coefficient": []
}

deep = {
    "Pixel Accuracy": [],
    "Mean IoU": [],
    "Dice Coefficient": []
}


In [45]:
# Đường dẫn đến thư mục chứa ảnh
image_folder = "test/img"

# Duyệt qua từng file trong thư mục
for file_name in os.listdir(image_folder):
    file_path = os.path.join(image_folder, file_name)

In [42]:
print(type(test[2][-4:]))
number = float(test[2][-4:])
number

<class 'str'>


0.8

In [25]:
# Gradio Interface
with gr.Blocks() as demo:
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Upload Image and Select Metric from Test folder ###")

            uploaded_image = gr.Image(type="filepath", label="Upload Image")

            output_choice = gr.Radio(
                choices= ['Segmentation Mask', 'Overlay Image'],
                value= 'Segmentation Mask',
                label= 'Output types'
            )

            metric_choice = gr.Radio(
                choices=["Pixel Accuracy", "Mean IoU", "Dice Coefficient"],
                value="Pixel Accuracy",
                label="Evaluation Metric",
            )

            submit_button = gr.Button("Submit")

            gt_class = gr.Textbox(label="Ground truth Class", interactive=False)

        with gr.Column():
            gr.Markdown("### Model Predictions and Metrics")
            with gr.Column():
                with gr.Column():
                    gr.Markdown("**FCN+Resnet50's Prediction**")
                    with gr.Row():
                      model_1_prediction = gr.Image(show_label=False)
                      with gr.Column():
                        model_1_metric = gr.Textbox(label="Metric Score", interactive=False)
                        model_1_class = gr.Textbox(label="Predicted Class", interactive=False)
                with gr.Column():
                    gr.Markdown("**UNet's Prediction**")
                    with gr.Row():
                      model_2_prediction = gr.Image(show_label=False)
                      with gr.Column():
                        model_2_metric = gr.Textbox(label="Metric Score", interactive=False)
                        model_2_class = gr.Textbox(label="Predicted Class", interactive=False)
                with gr.Column():
                    gr.Markdown("**DeepLab-V3's Prediction**")
                    with gr.Row():
                      model_3_prediction = gr.Image(show_label=False)
                      with gr.Column():
                        model_3_metric = gr.Textbox(label="Metric Score", interactive=False)
                        model_3_class = gr.Textbox(label="Predicted Class", interactive=False)

    # Link the submit button to the function
    submit_button.click(
        gr_wrapper,
        inputs=[uploaded_image, output_choice ,metric_choice],
        outputs=[
            gt_class,
            model_1_prediction, model_1_metric, model_1_class,
            model_2_prediction, model_2_metric, model_2_class,
            model_3_prediction, model_3_metric, model_3_class,
        ],
    )

# Launch the app
demo.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b4423cc59e83b2bf0c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b4423cc59e83b2bf0c.gradio.live


In [24]:
demo.close()

Closing server running on port: 7860
